# Fed-batch yeast: shaping the feed. Step or ramp profiles, monotone feeds, forced samples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/04_design_precision/c_profiles_and_sampling/tutorial.ipynb)

**The design space of `mbdoe_pp`**: what the optimiser is allowed to choose.

The best experiment depends on what the lab can do. `system['tvi']` says how an input may change
(`cvp`: steps or ramps; `stps`: how many levels, 1 for a constant input; `const`: free, only rising or only
falling; `offl`, `offt`: the smallest level change and the shortest time between changes), and
`system['tvo']` how samples may be taken (`sp` samples, `offt` apart, `samp_f` forced times).

You will design the same D-optimal run under five sets of rules and compare the information each
achieves: every rule has a price.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['model.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/04_design_precision/c_profiles_and_sampling/' + f, f)

### The process

A fed-batch baker's-yeast fermentation (Monod growth with maintenance):

$$r = \frac{\mu_{max}\, y_2}{K_s + y_2},\qquad \frac{dy_1}{dt} = (r - u_1 - m)\,y_1,\qquad
\frac{dy_2}{dt} = -\frac{r\,y_1}{Y_{xs}} + u_1 (u_2 - y_2)$$

Biomass `y1` and substrate `y2` are measured; the dilution rate `u1(t)` and the feed concentration
`u2(t)` can change during the run, the initial biomass `y10` is chosen per run. Four parameters:
`mu_max`, `Ks`, `Yxs`, `m`.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
import copy
from middoe.des_pp import mbdoe_pp

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta      = [0.25, 0.25, 0.88, 0.09]     # TRUE values: mu_max [1/h], Ks [g/L], Yxs [g/g], m [1/h]
theta_n    = [0.30, 0.30, 1.056, 0.108]   # starting guess (1.2 x true)
theta_mins = [0.125, 0.125, 0.44, 0.045]  # bounds
theta_maxs = [0.375, 0.375, 1.32, 0.135]
NAMES = ['mu_max', 'Ks', 'Yxs', 'm']

system = {
    't_s': [0, 40],          # 40-h fed batch
    't_r': 0.4 if QUICK else 0.16,   # time grid [h] (coarser in a quick check)
    't_d': 0,
    'tvi': {                 # inputs that change during the run
        'u1': {'min': 0.05, 'max': 0.2, 'cvp': 'CPF', 'stps': 5, 'const': 'rel', 'offl': 0.01, 'offt': 0.5},  # dilution rate [1/h], steps
        'u2': {'min': 5, 'max': 35, 'cvp': 'LPF', 'stps': 5, 'const': 'rel', 'offl': 1, 'offt': 0.5},        # feed substrate [g/L], ramps
    },
    'tii': {'y10': {'min': 1, 'max': 5.5}},                    # initial biomass [g/L]
    'tvo': {                 # measured outputs: 10 samples each, at least 0.5 h apart, noise sd 0.2 g/L
        'y1': {'meas': True, 'unc': 0.2, 'init': 0, 'sp': 10, 'samp_s': 1, 'samp_f': [], 'offt': 0.5},     # biomass
        'y2': {'meas': True, 'unc': 0.2, 'init': 0.01, 'sp': 10, 'samp_s': 1, 'samp_f': [], 'offt': 0.5},  # substrate
    },
    'tio': {},
}

models = {
    'can_m': ['MI'], 'krt': {'MI': 'pys'}, 'src': {'MI': str(HERE / 'model.py')}, 'creds': {'MI': ''},
    'theta': {'MI': list(theta_n)}, 't_l': {'MI': theta_mins}, 't_u': {'MI': theta_maxs},
    'mutation': {'MI': [True] * 4}, 'theta_names': {'MI': NAMES},
}
insilicos = {'tr_m': 'MI', 'theta': theta, 'errt': 'abs', 'plt': False,
             'prels': {'1': {'u1': 0.05, 'u2': 30, 'y10': 5.5}, '2': {'u1': 0.15, 'u2': 20, 'y10': 2.0}}}

iden_opt = {
    'meth': 'SLSQP', 'ms': False,           # local gradient-based optimiser, one start
    'ob': 'WLS',                            # weighted least squares
    'maxit': 60 if QUICK else 500, 'tol': 1e-8,
    'var-cov': 'M',                         # covariance from the Fisher information matrix
    'sens_m': 'central', 'eps': 1e-3,       # sensitivities by central differences, fixed step
    'c_plt': True, 'f_plt': True, 'plt_s': True, 'log': False,
}

In [ ]:
expera(system, models, insilicos, {}, expr=1)
expera(system, models, insilicos, {}, expr=2)
unc = uncert(parmest(system, models, iden_opt), system, models, iden_opt)   # prior information for the design

### Five rule sets

| variant | dilution rate `u1` | feed `u2` | sampling |
|---|---|---|---|
| **free** | 5 steps, free | 5 ramps, free | 10 free samples |
| **ramps** | 5 ramps | 5 ramps | 10 free samples |
| **monotone** | steps, only rising (`inc`) | ramps, only falling (`dec`) | 10 free samples |
| **constant** | one level (`stps=1`) | one level | 10 free samples |
| **few samples** | as *free* | as *free* | 5 samples, the last forced at 40 h, 2 h apart |

In [ ]:
def variant(name):
    s = copy.deepcopy(system)
    u1, u2 = s['tvi']['u1'], s['tvi']['u2']
    if name == 'ramps':
        u1['cvp'] = 'LPF'
    elif name == 'monotone':
        u1['const'], u2['const'] = 'inc', 'dec'
    elif name == 'constant':
        u1['stps'] = u2['stps'] = 1      # one level: the design picks a constant value
                                         #   ('no_CVP' is for simulation only, mbdoe_pp cannot design it)
    elif name == 'few samples':
        for y in s['tvo'].values():
            y.update(sp=5, samp_f=[40], offt=2)
    return s


des_opt = {'pp_ob': 'D', 'meth': 'PS', 'eps': 1e-3, 'priors': True, 'plt': True,
           'itr': {'maxpp': 15 if QUICK else 100, 'tolpp': 1, 'pps': 30}}

results = {}
for name in (['free', 'constant'] if QUICK else ['free', 'ramps', 'monotone', 'constant', 'few samples']):
    print(f'---------------- {name}')
    d = mbdoe_pp(des_opt, variant(name), models, round=2, num_parallel_runs=1 if QUICK else 4)
    results[name] = np.log10(abs(d['pp_obj']))     # pp_obj is det(FIM) for the D criterion
table = pd.DataFrame({'log10 det(FIM)': results})
table['orders of magnitude vs free'] = table.iloc[:, 0] - table.loc['free'].iloc[0]
table.round(2)

Reading the table: differences of log10 det(FIM) are orders of magnitude of information. With the
full budget (`QUICK = False`; in a quick check the optimiser stops long before convergence, so the
numbers mean little) a constant feed loses most; monotone feeds, which are often all a plant allows, lose little; fewer
samples cost information roughly in proportion.

**Try:** `stps` 3 or 8; a longer `offt` between switches; forced samples at the start
(`samp_f=[0]`); the E criterion.